# VinAI Translate baseline trên Kaggle

Trong **Notebook options**, bật GPU và Internet trước khi chạy. Notebook audit dữ liệu/token, chạy smoke test rồi mới cho phép full run. Output được ghi vào `/kaggle/working`, vì vậy cần **Save Version** để giữ kết quả.

In [ ]:
import os, subprocess, sys
from pathlib import Path
subprocess.run(["nvidia-smi"], check=False)
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Notebook options.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
REPO_URL = "https://github.com/PhThuan-tech/NLP.git"
PROJECT_DIR = Path("/kaggle/working/NLP")
if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-training.txt"], check=True)
print("Project:", Path.cwd())

In [ ]:
# Model public nên token không bắt buộc. Có thể tạo Kaggle Secret tên HF_TOKEN.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("Đã nạp HF_TOKEN từ Kaggle Secrets.")
except Exception:
    print("Không có HF_TOKEN; tiếp tục tải model public.")

In [ ]:
# Chỉ sửa ba biến này. MODE: zero_shot | lora | full
DIRECTION = "en2vi"
MODE = "lora"
RUN_FULL = False

if DIRECTION not in {"en2vi", "vi2en"}:
    raise ValueError("DIRECTION phải là en2vi hoặc vi2en")
if MODE not in {"zero_shot", "lora", "full"}:
    raise ValueError("MODE phải là zero_shot, lora hoặc full")
CONFIG = PROJECT_DIR / "configs" / "vinai" / f"{DIRECTION}_{MODE}.json"
DATA_DIR = PROJECT_DIR / "artifacts" / "processed_sklearn_seed67"
OUTPUT_DIR = Path("/kaggle/working/outputs") / f"vinai_{DIRECTION}_{MODE}"
required = [DATA_DIR / f"segments_dev_{DIRECTION}.jsonl"]
if MODE != "zero_shot":
    required.append(DATA_DIR / f"segments_train_{DIRECTION}.jsonl")
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Thiếu dữ liệu: " + ", ".join(missing))
print("Config:", CONFIG)
print("Output:", OUTPUT_DIR)

In [ ]:
def run_cli(command, *extra):
    args = [sys.executable, "-m", "src.vinai_baseline.cli", command,
            "--config", str(CONFIG), "--output-dir", str(OUTPUT_DIR), *map(str, extra)]
    print("RUN:", " ".join(args))
    subprocess.run(args, cwd=PROJECT_DIR, check=True)

run_cli("audit")

In [ ]:
COMMAND = "evaluate" if MODE == "zero_shot" else "train"
old_output = OUTPUT_DIR
OUTPUT_DIR = Path("/kaggle/working/smoke") / f"{DIRECTION}_{MODE}"
if MODE == "zero_shot":
    run_cli(COMMAND, "--max-validation-samples", "16")
else:
    run_cli(COMMAND, "--max-train-samples", "32", "--max-validation-samples", "16")
OUTPUT_DIR = old_output
print("Smoke test hoàn thành.")

In [ ]:
if RUN_FULL:
    run_cli(COMMAND)
else:
    print("Chưa chạy full. Kiểm tra smoke output, đặt RUN_FULL=True rồi chạy lại cell này.")

In [ ]:
import json
for name in ["resolved_config.json", "reports/token_audit_summary.json",
             "reports/validation_metrics.json", "run_manifest.json"]:
    path = OUTPUT_DIR / name
    if path.exists():
        print(f"\n--- {path} ---")
        print(path.read_text(encoding="utf-8")[:4000])
print("Sau khi hoàn tất, bấm Save Version để lưu /kaggle/working/outputs.")